# L1: 大语言模型基础

> 理解 LLM 是什么，如何工作

In [2]:
# === 环境设置 ===
import sys
import os
from pathlib import Path

# 自动找到项目根目录（包含 backend 文件夹的目录）
current_path = Path(os.getcwd())
project_path = None

# 向上搜索项目根目录
for path in [current_path] + list(current_path.parents):
    if (path / "backend").exists():
        project_path = str(path)
        break

# 如果没找到，尝试从 notebook 文件位置推断
if not project_path:
    notebook_path = Path(".").resolve()
    for path in [notebook_path] + list(notebook_path.parents):
        if (path / "backend").exists():
            project_path = str(path)
            break

# 添加到 Python 搜索路径
if project_path and project_path not in sys.path:
    sys.path.insert(0, project_path)

# 手动加载 .env 文件
env_file = Path(project_path) / ".env" if project_path else None
if env_file and env_file.exists():
    with open(env_file, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()

print(f"项目路径: {project_path}")
print(f"Python 版本: {sys.version.split()[0]}")
print(f"DEEPSEEK_API_KEY: {'已设置' if os.environ.get('DEEPSEEK_API_KEY') else '未设置'}")

# 验证 backend 模块
try:
    from backend import llm
    print("✓ backend 模块导入成功")
except ImportError as e:
    print(f"✗ backend 模块导入失败: {e}")

项目路径: c:\Users\zying\Desktop\pro_agent\agent-learning-project
Python 版本: 3.14.4
DEEPSEEK_API_KEY: 已设置
✓ backend 模块导入成功


## 1.1 什么是大语言模型 (LLM)?

**大语言模型 (Large Language Model)** 是基于深度学习的 AI 模型，通过在海量文本数据上训练，学会理解和生成人类语言。

### 核心特点

| 特点 | 说明 |
|------|------|
| **预训练** | 在大规模语料上学习语言规律 |
| **大参数** | 参数量从几十亿到万亿级别 |
| **通用能力** | 不需要针对特定任务训练 |
| **涌现能力** | 规模扩大后出现的新能力 |

## 1.2 核心概念

### Token 与 Tokenization

**Token** 是 LLM 处理文本的最小单位。

**Tokenization** 是将文本切分为 token 的**过程/动作**。

#### Token vs Tokenization 的区别

| | Token | Tokenization |
|---|---|---|
| **定义** | 单个文本单元 | 切分文本的过程/方法 |
| **本质** | 结果 | 动作 |
| **例子** | `[Hello]`, `[world]`, `[!]` | 把 "Hello world!" 切成这3个token |

#### 类比
- Token = 切好的苹果片
- Tokenization = 切苹果的动作

#### 常见 Tokenization 方法

```python
# 示例: "I love AI!"

# 1. Word-level:  [I] [love] [AI] [!]
# 2. Subword (BPE): [I] [love] [AI] [!]
# 3. Character: [I] [l][o][v][e] [A][I] [!]
```

现代 LLM（GPT、Claude等）主要使用 **Subword Tokenization**（如 BPE）。

### Context Window (上下文窗口)

每次调用 LLM 时能处理的最大 Token 数量。

### Temperature (温度参数)

控制输出随机性的参数，范围 [0, 2]。
- Temperature = 0: 确定性输出
- Temperature = 0.7: 平衡创造性和准确性
- Temperature = 1+: 更随机，适合创意写作

## 1.3 动手练习

### 练习 1: 基础问答

In [3]:
# 导入必要的模块
import asyncio
import os
from pathlib import Path
from backend.llm import LLMFactory, Message

# 读取 API key
def get_deepseek_key():
    project_path = Path(os.getcwd())
    for path in [project_path] + list(project_path.parents):
        env_file = path / ".env"
        if env_file.exists():
            with open(env_file, encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line.startswith("DEEPSEEK_API_KEY="):
                        return line.split("=", 1)[1].strip()
    return ""

async def basic_chat():
    # 创建 DeepSeek 客户端，直接传入 api_key
    api_key = get_deepseek_key()
    llm = LLMFactory.create("deepseek", api_key=api_key)

    # 发送简单问题
    response = await llm.chat(messages=[
        Message(role="user", content="什么是人工智能？用一句话回答。")
    ])

    print(f"回答: {response.content}")
    print(f"使用 Tokens: {response.usage.total_tokens}")

# 运行
await basic_chat()

回答: 人工智能是让机器模拟人类智能行为的技术，使其能学习、推理、感知和决策。
使用 Tokens: 31


### 练习 2: 测试温度参数

In [4]:
async def test_temperature():
    # 读取 API key 并创建客户端
    api_key = get_deepseek_key()
    llm = LLMFactory.create("deepseek", api_key=api_key)
    question = "用三个词描述夏天"

    for temp in [0.0, 0.7, 1.5]:
        response = await llm.chat(
            messages=[Message(role="user", content=question)],
            temperature=temp
        )
        print(f"Temperature {temp}: {response.content}")

await test_temperature()

Temperature 0.0: 炙热、蝉鸣、西瓜。
Temperature 0.7: 炙热、蝉鸣、悠长。
Temperature 1.5: 烈日，蝉鸣，融化的冰棍。


### 练习 3: 多轮对话

In [ ]:
async def multi_turn():
    # 读取 API key 并创建客户端
    api_key = get_deepseek_key()
    llm = LLMFactory.create("deepseek", api_key=api_key)

    messages = [
        Message(role="system", content="你是一个友好的数学老师"),
    ]

    # 第一轮
    messages.append(Message(role="user", content="你好！我是学生小明"))
    response1 = await llm.chat(messages=messages)
    print(f"老师: {response1.content}")
    
    messages.append(Message(role="assistant", content=response1.content))

    # 第二轮
    messages.append(Message(role="user", content="请问 7 + 8 等于多少？"))
    response2 = await llm.chat(messages=messages)
    print(f"老师: {response2.content}")
    # 这里没有做任何窗口管理的优化
    print(messages)

await multi_turn()

老师: 你好呀，小明！我是你的数学老师，很高兴见到你！有什么数学问题或者学习上的疑惑需要我帮忙吗？我们一起努力，数学其实可以很有趣哦～ 😊
老师: 哈哈，小明，这个问题很简单哦！  
7 + 8 等于 **15**。  
你可以这样想：7 加上 8，可以先把 7 凑成 10（需要加 3），那么 8 就剩下 5，10 + 5 = 15。  
或者直接数手指：7、8、9、10、11、12、13、14、15！  
你学会了吗？ 😊
[Message(role='system', content='你是一个友好的数学老师', tool_call_id=None, tool_calls=None, metadata=None), Message(role='user', content='你好！我是学生小明', tool_call_id=None, tool_calls=None, metadata=None), Message(role='assistant', content='你好呀，小明！我是你的数学老师，很高兴见到你！有什么数学问题或者学习上的疑惑需要我帮忙吗？我们一起努力，数学其实可以很有趣哦～ 😊', tool_call_id=None, tool_calls=None, metadata=None), Message(role='user', content='请问 7 + 8 等于多少？', tool_call_id=None, tool_calls=None, metadata=None)]


## 1.4 查看数据模型

In [ ]:
from email import message
from backend.llm.models import Message

# 查看 Message 类的字段
print("Message 类的字段:")
for field_name in Message.model_fields:
    print(f"  - {field_name}")
print(Message)

Message 类的字段:
  - role
  - content
  - tool_call_id
  - tool_calls
  - metadata
<class 'backend.llm.models.Message'>


## 1.5 查看可用的 LLM 提供商

In [6]:
from backend.llm import LLMFactory

print("可用的 LLM 提供商:")
for provider in LLMFactory.list_providers():
    print(f"  - {provider}")

可用的 LLM 提供商:
  - openai
  - anthropic
  - ollama
  - glm
  - deepseek


## 1.6 深入理解上下文窗口 (Context Window)

### 什么是上下文窗口？

上下文窗口 = LLM 一次性"能看到"的最大文本量。

**类比**：就像考试时的"开卷"范围，你只能翻看指定页数内的内容。

### 面试知识点

| 知识点 | 说明 |
|--------|------|
| **Token vs 字符** | 1 Token ≈ 0.75 个英文词，中文约 2-3 字符/Token |
| **输入 + 输出都算** | 请求和响应的总和不能超过窗口大小 |
| **超窗口处理** | 截断旧内容、摘要压缩、RAG 检索 |
| **不同模型窗口** | GPT-4: 128K, Claude: 200K, DeepSeek: 32K |
| **与 Memory 的关系** | Memory 是解决窗口有限的核心策略 |

In [7]:
# 演示 1: 观察 Token 使用情况
import asyncio
import os
from pathlib import Path
from backend.llm import LLMFactory, Message

async def demo_token_usage():
    """演示不同输入长度对 Token 使用的影响"""
    api_key = get_deepseek_key()
    llm = LLMFactory.create("deepseek", api_key=api_key)
    
    # 测试不同长度的输入
    test_cases = [
        ("短文本", "你好"),
        ("中文本", "请用三句话介绍人工智能的发展历史"),
        ("长文本", "请详细分析以下问题：什么是人工智能？它的发展历程如何？有哪些主要分支？请从机器学习、深度学习、强化学习等角度详细阐述。"),
    ]
    
    print("=" * 60)
    print("Token 使用情况演示")
    print("=" * 60)
    
    for name, content in test_cases:
        response = await llm.chat(
            messages=[Message(role="user", content=content)]
        )
        print(f"\n{name}:")
        print(f"  输入: {content[:30]}...")
        print(f"  Prompt Tokens: {response.usage.prompt_tokens}")
        print(f"  Completion Tokens: {response.usage.completion_tokens}")
        print(f"  总计: {response.usage.total_tokens}")

await demo_token_usage()

Token 使用情况演示

短文本:
  输入: 你好...
  Prompt Tokens: 5
  Completion Tokens: 30
  总计: 35

中文本:
  输入: 请用三句话介绍人工智能的发展历史...
  Prompt Tokens: 12
  Completion Tokens: 96
  总计: 108

长文本:
  输入: 请详细分析以下问题：什么是人工智能？它的发展历程如何？有哪些...
  Prompt Tokens: 35
  Completion Tokens: 2728
  总计: 2763


In [9]:
# 演示 2: 模拟上下文窗口溢出场景
async def demo_context_overflow():
    """
    演示：当对话历史过长时会发生什么
    
    现实中，超窗口会被 API 拒绝，但这里我们模拟概念
    """
    print("=" * 60)
    print("上下文窗口溢出演示")
    print("=" * 60)
    
    # 模拟一个有限的上下文窗口
    CONTEXT_WINDOW_SIZE = 500  # 假设窗口大小为 1000 tokens
    
    messages = []
    total_tokens = 0
    
    # 模拟多轮对话
    for i in range(10):
        user_msg = f"这是第 {i+1} 轮对话，我问一个问题" * 10
        messages.append(Message(role="user", content=user_msg))
        
        # 估算 token（粗略计算：中文约 2 字符/token）
        estimated_tokens = len(user_msg) // 2
        total_tokens += estimated_tokens
        
        status = "✓ 正常" if total_tokens < CONTEXT_WINDOW_SIZE else "⚠ 超出窗口!"
        print(f"轮次 {i+1}: 累计 ~{total_tokens} tokens {status}")
        
        if total_tokens >= CONTEXT_WINDOW_SIZE:
            print(f"\n❌ 第 {i+1} 轮超出上下文窗口!")
            print(f"   需要策略：截断旧消息 / 压缩摘要 / 使用 Memory")
            break

await demo_context_overflow()

上下文窗口溢出演示
轮次 1: 累计 ~80 tokens ✓ 正常
轮次 2: 累计 ~160 tokens ✓ 正常
轮次 3: 累计 ~240 tokens ✓ 正常
轮次 4: 累计 ~320 tokens ✓ 正常
轮次 5: 累计 ~400 tokens ✓ 正常
轮次 6: 累计 ~480 tokens ✓ 正常
轮次 7: 累计 ~560 tokens ⚠ 超出窗口!

❌ 第 7 轮超出上下文窗口!
   需要策略：截断旧消息 / 压缩摘要 / 使用 Memory


### Context Window 与 Memory 的关系

```
┌─────────────────────────────────────────────────────────────┐
│                     Context Window (有限)                    │
│  ┌───────────────────────────────────────────────────────┐  │
│  │ 对话历史 + 知识检索 + System Prompt                    │  │
│  └───────────────────────────────────────────────────────┘  │
└─────────────────────────────────────────────────────────────┘
                            ↓
                    超过限制会报错
                            ↓
        ┌───────────────────┴───────────────────┐
        │           Memory 解决方案              │
        ├───────────────────┼───────────────────┤
        │                   │                   │
    ┌───▼────┐      ┌──────▼──────┐    ┌─────▼─────┐
    │短期记忆 │      │  长期记忆    │    │   RAG     │
    │(Session)│      │(Vector DB)  │    │(检索增强) │
    │保留最近 │      │语义搜索旧对话│    │外部知识库 │
    │N 条消息 │      │             │    │           │
    └─────────┘      └─────────────┘    └───────────┘
```

**核心思想**：Memory 是在有限的上下文窗口中"智能选择"放入哪些内容。

### 常见面试题

**Q: 如何处理超长对话？**
- A: 滑动窗口（保留最近N条）、摘要压缩、RAG检索

**Q: 上下文窗口和 Memory 有什么区别？**
- A: 窗口是模型的技术限制，Memory是工程策略

**Q: Token 和字符有什么关系？**
- A: 英文约4字符/token，中文约2-3字符/token（取决于tokenizer）

**Q: 为什么不同模型窗口大小不同？**
- A: 与模型架构、训练目标、推理成本相关。更大窗口需要更多计算资源

**Q: 什么是 KV Cache？**
- A: 推理加速技术。缓存自注意力层的 Key 和 Value 矩阵，避免重复计算。
  - 生成阶段，每个新 token 只需计算与之前所有 token 的注意力
  - 没有 KV Cache：O(n²) 重复计算
  - 有 KV Cache：只计算新 token 的部分
  - 权衡：显存占用随序列长度增长

**Q: 什么是 Middle Loss？**
- A: 一种训练策略，不仅预测下一个 token，也在序列中间位置计算损失。
  - 传统 LLM 只在最后一个 token 计算损失
  - Middle Loss 在序列中间多个位置计算损失
  - 好处：提升模型对长序列中间内容的理解能力

**Q: 什么是长上下文模型（Long Context Model）？**
- A: 支持超长输入的 LLM，通常指窗口 ≥ 32K，甚至 1M+ 的模型。
  - **技术手段**：RoPE 位置编码、ALiBi、FlashAttention、GQA
  - **代表模型**：Claude 200K、GPT-4 Turbo 128K、Gemini 1M、GLM-1M
  - **挑战**：推理成本高、显存占用大、"大海捞针"能力下降

**Q: RAG vs 长上下文，如何选择？**

| 维度 | RAG | 长上下文 |
|------|-----|---------|
| **成本** | 低（只检索相关片段） | 高（处理全部内容） |
| **准确性** | 高（精确定位） | 可能遗漏细节 |
| **实时性** | 需更新向量库 | 直接输入，无延迟 |
| **适用场景** | 知识库、文档问答 | 长文档分析、代码理解 |

**选择策略**：
- 文档频繁更新 → RAG
- 需要跨章节推理 → 长上下文
- 成本敏感 → RAG
- 准确性要求高 → RAG + 长上下文（先用 RAG 检索，再用长上下文模型处理）

**Q: 注意力计算的复杂度是多少？**
- A: 标准 Transformer 自注意力复杂度为 **O(n²)**，其中 n 是序列长度。
  - **原因**：每个 token 需要与所有其他 token 计算注意力权重
  - Q × K^T 生成 n × n 的注意力矩阵
  - **影响**：序列长度翻倍，计算量翻四倍
  - **优化方案**：
    - FlashAttention：通过分块计算减少显存访问，提升实际速度
    - GQA（Grouped Query Attention）：多查询共享 Key/Value，减少计算量
    - Sliding Window：只计算局部窗口内的注意力
    - Sparse Attention：稀疏注意力模式（如 Longformer）

**Q: 位置编码与上下文长度的关系？**
- A: 位置编码决定模型能处理的最大序列长度。
  - **绝对位置编码**（如 BERT）：训练时确定最大长度，超出无法处理
  - **相对位置编码**（如 T5）：基于相对距离，理论上可外推到更长序列
  - **RoPE（旋转位置编码）**（GPT-3/LLaMA/DeepSeek 使用）：
    - 通过旋转矩阵编码位置信息
    - 理论上支持外推到训练时未见的长度
    - 实际中超长时性能下降，需用 YaRN、NTK-aware 等插值技术
  - **ALiBi**：不训练位置编码，用线性偏置，外推能力强
  - **关键结论**：扩展上下文窗口不仅是模型参数问题，更依赖位置编码设计